In [1]:
# 05_xgboost_test_2.ipynb
# 实验：XGBoost 500棵树 + max_depth调优

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, recall_score, f1_score
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("05_xgboost_test_2: 500棵树 + max_depth调优")
print("="*60)

# 1. 加载数据
print("\n1. 加载数据...")
train = pd.read_csv('../data/train.csv')
print(f"训练集形状: {train.shape}")

# 2. 准备数据
print("\n2. 准备数据...")
X = train.drop(['id', 'diagnosed_diabetes'], axis=1)
y = train['diagnosed_diabetes']

# 编码类别特征
categorical_cols = ['gender', 'ethnicity', 'education_level', 
                   'income_level', 'smoking_status', 'employment_status']
X_encoded = pd.get_dummies(X, columns=categorical_cols)
print(f"编码后特征数: {X_encoded.shape[1]}")

# 分割数据
X_train, X_val, y_train, y_val = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)
print(f"训练集: {X_train.shape}, 验证集: {X_val.shape}")

# 3. 计算类别权重
n_negative = sum(y_train == 0)
n_positive = sum(y_train == 1)
scale_pos_weight = n_negative / n_positive
print(f"\nscale_pos_weight: {scale_pos_weight:.3f}")

# 4. 之前的实验结果
prev_exp = {
    'name': '04_xgboost_test_1 (300 trees)',
    'auc': 0.7255,
    'recall_0': 0.695,
    'recall_1': 0.631,
    'params': {'n_estimators': 300, 'max_depth': 6}
}

print(f"\n之前的实验 [{prev_exp['name']}] 结果:")
print(f"- AUC: {prev_exp['auc']}")
print(f"- 0类召回率: {prev_exp['recall_0']}")
print(f"- 1类召回率: {prev_exp['recall_1']}")
print(f"- 召回率gap: {prev_exp['recall_1'] - prev_exp['recall_0']:.3f}")

# 5. 尝试不同的max_depth
print("\n5. 尝试不同的max_depth参数...")

depths_to_try = [4, 6, 8, 10]  # 尝试4种深度
results_dict = {}

plt.figure(figsize=(15, 10))

for idx, depth in enumerate(depths_to_try):
    print(f"\n▶ 训练 max_depth={depth}...")
    
    model = xgb.XGBClassifier(
        n_estimators=500,              # 增加到500
        learning_rate=0.1,
        max_depth=depth,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        eval_metric='auc',
        use_label_encoder=False,
        early_stopping_rounds=50
    )
    
    # 训练
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    # 评估
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    y_pred = model.predict(X_val)
    
    auc = roc_auc_score(y_val, y_pred_proba)
    recall_0 = recall_score(y_val, y_pred, pos_label=0)
    recall_1 = recall_score(y_val, y_pred, pos_label=1)
    f1_0 = f1_score(y_val, y_pred, pos_label=0)
    f1_1 = f1_score(y_val, y_pred, pos_label=1)
    
    results_dict[depth] = {
        'model': model,
        'auc': auc,
        'recall_0': recall_0,
        'recall_1': recall_1,
        'f1_0': f1_0,
        'f1_1': f1_1,
        'results': model.evals_result()
    }
    
    print(f"  AUC: {auc:.4f} | 0类召回: {recall_0:.3f} | 1类召回: {recall_1:.3f} | F1平均: {(f1_0+f1_1)/2:.3f}")
    
    # 绘制学习曲线（子图）
    plt.subplot(2, 2, idx+1)
    results = model.evals_result()
    plt.plot(results['validation_0']['auc'], label=f'max_depth={depth}')
    plt.axhline(y=prev_exp['auc'], color='gray', linestyle='--', alpha=0.5, label='Previous best')
    plt.xlabel('Rounds')
    plt.ylabel('AUC')
    plt.title(f'Learning Curve (max_depth={depth})')
    plt.legend()
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../logs/05_xgboost_test_2_learning_curves.png', dpi=100, bbox_inches='tight')
plt.show()

# 6. 找出最佳depth
print("\n6. 结果对比...")

best_depth = max(results_dict, key=lambda d: results_dict[d]['auc'])
best_model = results_dict[best_depth]['model']
best_auc = results_dict[best_depth]['auc']
best_recall_0 = results_dict[best_depth]['recall_0']
best_recall_1 = results_dict[best_depth]['recall_1']

print("\n所有max_depth结果:")
print("-" * 70)
print(f"{'depth':<8} {'AUC':<8} {'Recall_0':<10} {'Recall_1':<10} {'F1_0':<8} {'F1_1':<8} {'F1_avg':<8}")
print("-" * 70)
for depth in depths_to_try:
    r = results_dict[depth]
    f1_avg = (r['f1_0'] + r['f1_1']) / 2
    print(f"{depth:<8} {r['auc']:<8.4f} {r['recall_0']:<10.3f} {r['recall_1']:<10.3f} {r['f1_0']:<8.3f} {r['f1_1']:<8.3f} {f1_avg:<8.3f}")
print("-" * 70)
print(f"最佳depth: {best_depth} (AUC={best_auc:.4f})")

# 7. 对比之前的实验
print(f"\n7. 与之前的实验对比:")
print(f"- 之前 ({prev_exp['name']}): AUC={prev_exp['auc']:.4f}, 0类召回={prev_exp['recall_0']:.3f}, 1类召回={prev_exp['recall_1']:.3f}")
print(f"- 当前 (depth={best_depth}): AUC={best_auc:.4f}, 0类召回={best_recall_0:.3f}, 1类召回={best_recall_1:.3f}")
print(f"- AUC提升: +{best_auc - prev_exp['auc']:.4f}")

# 8. 阈值优化分析（用最佳模型）
print("\n8. 阈值优化分析...")

y_pred_proba_best = best_model.predict_proba(X_val)[:, 1]

thresholds = [0.3, 0.4, 0.45, 0.5, 0.55, 0.6, 0.7]
print("\n不同阈值下的表现:")
print("-" * 75)
print(f"{'阈值':<8} {'0类召回':<10} {'1类召回':<10} {'gap':<8} {'F1_0':<8} {'F1_1':<8} {'F1_avg':<8}")
print("-" * 75)

best_f1_avg = 0
best_threshold = 0.5

for thresh in thresholds:
    y_pred_thresh = (y_pred_proba_best >= thresh).astype(float)
    r0 = recall_score(y_val, y_pred_thresh, pos_label=0)
    r1 = recall_score(y_val, y_pred_thresh, pos_label=1)
    f1_0 = f1_score(y_val, y_pred_thresh, pos_label=0)
    f1_1 = f1_score(y_val, y_pred_thresh, pos_label=1)
    f1_avg = (f1_0 + f1_1) / 2
    gap = r1 - r0
    print(f"{thresh:<8.2f} {r0:<10.3f} {r1:<10.3f} {gap:<8.3f} {f1_0:<8.3f} {f1_1:<8.3f} {f1_avg:<8.3f}")
    
    if f1_avg > best_f1_avg:
        best_f1_avg = f1_avg
        best_threshold = thresh

print("-" * 75)
print(f"最佳阈值: {best_threshold} (平均F1={best_f1_avg:.3f})")

# 9. 特征重要性可视化
print("\n9. 特征重要性分析...")

plt.figure(figsize=(12, 8))
importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

plt.barh(importance['feature'], importance['importance'])
plt.xlabel('Importance')
plt.title(f'Top 20 Feature Importance (max_depth={best_depth})')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../logs/05_xgboost_test_2_feature_importance.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nTop 10最重要特征:")
print(importance.head(10).to_string(index=False))

# 10. 保存模型和文件
print("\n10. 保存模型和提交文件...")

# 保存模型
model_path = f'../models/05_xgboost_test_2_depth{best_depth}.pkl'
joblib.dump(best_model, model_path)
print(f"模型已保存到 {model_path}")

# 生成提交文件
test = pd.read_csv('../data/test.csv')
X_test = test.drop(['id'], axis=1)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols)

# 确保列一致
missing_cols = set(X_train.columns) - set(X_test_encoded.columns)
for col in missing_cols:
    X_test_encoded[col] = 0
X_test_encoded = X_test_encoded[X_train.columns]

test_pred = best_model.predict_proba(X_test_encoded)[:, 1]

submission = pd.DataFrame({
    'id': test['id'],
    'diagnosed_diabetes': test_pred
})
submission_path = '../submissions/05_xgboost_test_2.csv'
submission.to_csv(submission_path, index=False)
print(f"提交文件已保存到 {submission_path}")

05_xgboost_test_2: 500棵树 + max_depth调优

1. 加载数据...
训练集形状: (700000, 26)

2. 准备数据...
编码后特征数: 42
训练集: (560000, 42), 验证集: (140000, 42)

scale_pos_weight: 0.604

之前的实验 [04_xgboost_test_1 (300 trees)] 结果:
- AUC: 0.7255
- 0类召回率: 0.695
- 1类召回率: 0.631
- 召回率gap: -0.064

5. 尝试不同的max_depth参数...

▶ 训练 max_depth=4...
  AUC: 0.7257 | 0类召回: 0.698 | 1类召回: 0.628 | F1平均: 0.648

▶ 训练 max_depth=6...
  AUC: 0.7264 | 0类召回: 0.692 | 1类召回: 0.635 | F1平均: 0.650

▶ 训练 max_depth=8...
  AUC: 0.7253 | 0类召回: 0.687 | 1类召回: 0.638 | F1平均: 0.650

▶ 训练 max_depth=10...
  AUC: 0.7228 | 0类召回: 0.681 | 1类召回: 0.640 | F1平均: 0.648


<Figure size 1500x1000 with 4 Axes>


6. 结果对比...

所有max_depth结果:
----------------------------------------------------------------------
depth    AUC      Recall_0   Recall_1   F1_0     F1_1     F1_avg  
----------------------------------------------------------------------
4        0.7257   0.698      0.628      0.603    0.693    0.648   
6        0.7264   0.692      0.635      0.603    0.698    0.650   
8        0.7253   0.687      0.638      0.601    0.699    0.650   
10       0.7228   0.681      0.640      0.598    0.698    0.648   
----------------------------------------------------------------------
最佳depth: 6 (AUC=0.7264)

7. 与之前的实验对比:
- 之前 (04_xgboost_test_1 (300 trees)): AUC=0.7255, 0类召回=0.695, 1类召回=0.631
- 当前 (depth=6): AUC=0.7264, 0类召回=0.692, 1类召回=0.635
- AUC提升: +0.0009

8. 阈值优化分析...

不同阈值下的表现:
---------------------------------------------------------------------------
阈值       0类召回       1类召回       gap      F1_0     F1_1     F1_avg  
---------------------------------------------------------------------------
0

<Figure size 1200x800 with 1 Axes>


Top 10最重要特征:
feature  importance
           family_history_diabetes    0.766882
                               age    0.031437
physical_activity_minutes_per_week    0.029840
                     triglycerides    0.010445
                               bmi    0.008610
                   ldl_cholesterol    0.007310
            cardiovascular_history    0.006974
                        diet_score    0.005428
                   hdl_cholesterol    0.005418
                        heart_rate    0.005087

10. 保存模型和提交文件...
模型已保存到 ../models/05_xgboost_test_2_depth6.pkl
提交文件已保存到 ../submissions/05_xgboost_test_2.csv


============================================================
05_xgboost_test_2 总结报告
============================================================

[Experiment Information]
File Name: 05_xgboost_test_2
Date: 2026-03-16 00:26
Model: XGBoost Classifier

[Parameters Tested]
- n_estimators: 500 (fixed)
- learning_rate: 0.1 (fixed)
- max_depth: tested [4, 6, 8, 10]
- scale_pos_weight: 0.604

[Best Parameters]
- max_depth: 6
- n_estimators: 500
- learning_rate: 0.1

[Results Comparison]
Previous Best (04_xgboost_test_1, depth=6, 300 trees):
- AUC: 0.7255
- Class 0 Recall: 0.695
- Class 1 Recall: 0.631
- Recall Gap: -0.064

Current Best (depth=6, 500 trees):
- AUC: 0.7264
- Class 0 Recall: 0.692
- Class 1 Recall: 0.635
- Recall Gap: -0.057

[Improvements]
- AUC Improvement: +0.0009
- Gap Change: -0.064 → -0.057

[Key Findings]
1. Best max_depth:6
2. Model Convergence:Converged
3. Optimal Threshold:0.45

[Next Steps]
- Experiment with learning_rate
- Focus on feature engineering
- Consider feature engineering based on top features